## 03 · The Reward Hack

The rulebook follows instructions written in advance. This notebook introduces a different kind of operator: an AI agent that starts knowing nothing and learns by trial and error. It runs thousands of simulated days, receives a score after every decision, and gradually shifts toward decisions that score well.

The catch is that it learns whatever the score rewards, not what we meant. This notebook trains the agent on the scoring rule used in the original version of this project, and shows what it learned.

### The original scoring rule

Each hour, the agent loses points for:

- **fuel burned** by the generator
- **unmet demand**, at $2 per kWh not delivered
- **battery stress** when the battery runs very low
- a small **$0.10 penalty** for running diesel while solar alone could cover demand

One gap matters: households cut off with the `shed_load` choice are not counted as unmet demand. Under this rule, deliberately cutting power costs nothing.

In [ ]:
# Train a PPO agent on the original reward (or load it if already trained), using the original settings
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from minigrid_mdp.env.legacy_reward import V1RewardEnv

MODELS_DIR = Path("..") / "models"
PPO_SETTINGS = dict(n_steps=512, batch_size=64, n_epochs=10, gamma=0.99,
                    learning_rate=3e-4, ent_coef=0.01, seed=42)

def train_or_load(env_cls, path, timesteps=200_000):
    if path.exists():
        return PPO.load(path)
    train_env = make_vec_env(env_cls, n_envs=1, wrapper_class=Monitor)
    model = PPO("MlpPolicy", train_env, verbose=0, **PPO_SETTINGS)
    model.learn(total_timesteps=timesteps)
    model.save(path)
    return model

hack_model = train_or_load(V1RewardEnv, MODELS_DIR / "ppo_v1_reward.zip")